# ResNet34-UNet + H95 — A100 Colab Training

**Target hardware:** NVIDIA A100 40 GB (Colab High-RAM runtime). Falls back to V100/L4 with reduced batch and FP16. T4 is rejected.

**Approach:** Drives the existing `src/` pipeline in this repo (see `instructions/IMPLEMENTATION_PLAN.md`). No source edits — this notebook mounts Drive, copies the code locally, builds a Colab-tuned config, and calls each pipeline stage in process.

**Inputs:**
- DocTamper LMDBs at `/content/drive/MyDrive/usth/Final_Project/dtset/archive`
- Centralised tamper metadata at `/content/drive/MyDrive/usth/Final_Project/Approach/tampering_types`
- Code mirror at `/content/drive/Othercomputers/MSI/Approach/RESNET34/Train_resnet_server`

**Outputs:** stream straight to `/content/drive/Othercomputers/MSI/results/doctamper_resnet34_h95_35epochs_comparison/` so they appear on the MSI machine via Drive for Desktop.

**Stage toggles** live in section 3 — flip individual stages on/off independently.

## 1. Mount Drive & validate paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

CODE_MIRROR  = Path('/content/drive/Othercomputers/MSI/Approach/RESNET34/Train_resnet_server')
DATA_ROOT    = Path('/content/drive/MyDrive/usth/Final_Project/dtset/archive')
TAMPER_META  = Path('/content/drive/MyDrive/usth/Final_Project/Approach/tampering_types')
RESULTS_ROOT = Path('/content/drive/Othercomputers/MSI/results')

for label, p in [('code mirror', CODE_MIRROR), ('data root', DATA_ROOT), ('tamper metadata', TAMPER_META)]:
    assert p.exists(), f'{label} not found: {p}'
    print(f'OK  {label}: {p}')

RESULTS_ROOT.parent.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print(f'OK  results root: {RESULTS_ROOT}')

## 2. GPU detection — warn & fall back, do not hard-assert

Decides `amp_dtype` and `batch_size_candidates` from the GPU Colab actually allocated. Picked up by section 7 when the config is built.

In [ ]:
import torch

assert torch.cuda.is_available(), 'CUDA is not available — switch to a GPU runtime'
gpu_name       = torch.cuda.get_device_name(0)
total_vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_supported = torch.cuda.is_bf16_supported()
print(f'GPU:  {gpu_name}')
print(f'VRAM: {total_vram_gb:.2f} GB')
print(f'BF16: {bf16_supported}')

if 'A100' in gpu_name:
    AMP_DTYPE        = 'bf16'
    BATCH_CANDIDATES = [56, 60, 64, 72, 80, 96]
    MEMORY_FRACTION  = 0.85
    print('✓ A100 detected — BF16, candidates [16..64], memory_fraction 0.80')
elif 'L4' in gpu_name:
    # L4 is Ada Lovelace (compute 8.9): BF16 native, 22.5 GB VRAM.
    # Empirically ResNet34-UNet @ 512² uses ~340 MB/sample in the autotune probe,
    # so batch=48 ≈ 16 GB, batch=64 ≈ 22 GB — push to the largest that fits at 0.85.
    AMP_DTYPE        = 'bf16' if bf16_supported else 'fp16'
    BATCH_CANDIDATES = [48, 50, 56, 64]
    MEMORY_FRACTION  = 0.85
    print(f'✓ L4 detected ({total_vram_gb:.1f} GB) — {AMP_DTYPE.upper()}, candidates [24..64], memory_fraction 0.85')
elif 'V100' in gpu_name:
    AMP_DTYPE        = 'fp16'
    MEMORY_FRACTION  = 0.80
    if total_vram_gb >= 24:
        BATCH_CANDIDATES = [16, 24, 32, 48, 64]
        print(f'⚠  V100-32GB detected — FP16, candidates [16..64]')
    else:
        BATCH_CANDIDATES = [8, 16, 24, 32]
        print(f'⚠  V100-16GB detected — FP16, candidates [8..32]')
elif 'T4' in gpu_name:
    raise RuntimeError(
        f'{gpu_name} only has ~16 GB — insufficient for ResNet34-UNet at 512x512. '
        'Disconnect and request a larger GPU.'
    )
else:
    AMP_DTYPE        = 'bf16' if bf16_supported else 'fp16'
    BATCH_CANDIDATES = [8, 16, 24, 32]
    MEMORY_FRACTION  = 0.80
    print(f'⚠  Unknown GPU {gpu_name} — defensive defaults: {AMP_DTYPE}, candidates [8..32]')

## 3. Pipeline switches — EDIT HERE

Each stage is independent. A stage that depends on `checkpoint` (eval / failure / tamper) will use one of the following, in order:
1. The `checkpoint` produced by section 9 if `RUN_TRAIN = True` ran in this session.
2. `CHECKPOINT_OVERRIDE` if you set it below.
3. `<RESULTS_ROOT>/<run_dir>/checkpoints/best_model.pth` auto-discovered from disk.

If none of those exist, the post-train stages will raise a clear error rather than running silently against a random model.

In [ ]:
# ─── Stage on/off ────────────────────────────────────────────────────────────
RUN_SMOKE   = True   # § 8  — 2-sample read of train + every test LMDB
RUN_TRAIN   = True   # § 9  — full training loop
RUN_EVAL    = True   # § 10 — per-test-set metrics + threshold sweep + diagnostic panels
RUN_FAILURE = True   # § 11 — worst-K failure case analysis + summaries
RUN_TAMPER  = True   # § 12 — per-tamper-type analysis

# ─── Run overrides ───────────────────────────────────────────────────────────
EPOCHS_OVERRIDE     = 1   # int  → overrides training.epochs in the YAML; None keeps the config value (35)
CHECKPOINT_OVERRIDE = None   # str  → absolute path to a *.pth used by eval/failure/tamper when RUN_TRAIN is False

# ─── Autotune overrides (None → keep GPU-detected values from § 2) ───────────
# Bump these when § 9's autotune picks too low.
# Heuristic: ResNet34-UNet @ 512² peaks at ~340 MB/sample in the probe.
# Real training adds ~500 MB of optimizer + prefetch overhead on top.
#   22 GB GPU (L4):    [24, 32, 48, 56, 64] @ 0.85 → picks 48
#   22 GB, more risk:  [32, 48, 56, 64]     @ 0.88 → picks 56
#   40 GB (A100):      [16, 24, 32, 48, 64] @ 0.80 → picks 64
#   40 GB, aggressive: [48, 64, 80, 96]     @ 0.86 → picks 80–96 depending on session
BATCH_CANDIDATES_OVERRIDE = None   # list[int]   e.g. [32, 48, 56, 64]
MEMORY_FRACTION_OVERRIDE  = None   # float 0..1  e.g. 0.88

print(f'RUN_SMOKE   = {RUN_SMOKE}')
print(f'RUN_TRAIN   = {RUN_TRAIN}')
print(f'RUN_EVAL    = {RUN_EVAL}')
print(f'RUN_FAILURE = {RUN_FAILURE}')
print(f'RUN_TAMPER  = {RUN_TAMPER}')
print(f'EPOCHS_OVERRIDE           = {EPOCHS_OVERRIDE}')
print(f'CHECKPOINT_OVERRIDE       = {CHECKPOINT_OVERRIDE}')
print(f'BATCH_CANDIDATES_OVERRIDE = {BATCH_CANDIDATES_OVERRIDE}')
print(f'MEMORY_FRACTION_OVERRIDE  = {MEMORY_FRACTION_OVERRIDE}')

## 4. Install dependencies

In [ ]:
!pip -q install -r /content/drive/Othercomputers/MSI/Approach/RESNET34/Train_resnet_server/requirements.txt

## 5. Copy the repo to local Colab disk (guarded, one-shot)

Uses `cp -r` once, then prunes heavy result folders. **Does not** use `rsync --delete` — local debugging edits in `/content/Train_resnet_server/src/` survive, but you must push them back to the Drive mirror manually before the session ends.

In [ ]:
import subprocess

LOCAL_REPO = Path('/content/Train_resnet_server')
if not LOCAL_REPO.exists():
    print(f'Copying code mirror → {LOCAL_REPO} ...')
    subprocess.run(['cp', '-r', str(CODE_MIRROR), '/content/'], check=True)
    for sub in ['res', 'runs', '.codex']:
        target = LOCAL_REPO / sub
        if target.exists():
            subprocess.run(['rm', '-rf', str(target)], check=True)
            print(f'  pruned {target}')
else:
    print(f'Local repo already present: {LOCAL_REPO}')

print('Local src contents:')
print(sorted(p.name for p in (LOCAL_REPO / 'src').iterdir()))

## 6. Make `src` importable

In [ ]:
import os, sys
os.chdir(LOCAL_REPO)
if str(LOCAL_REPO) not in sys.path:
    sys.path.insert(0, str(LOCAL_REPO))
print(f'cwd:         {os.getcwd()}')
print(f'sys.path[0]: {sys.path[0]}')

## 7. Build the A100-tuned config

Loads the shipped `configs/resnet_h95_config.yaml`, overrides Drive paths + GPU knobs + (optionally) `training.epochs`, and writes `configs/resnet_h95_a100.yaml`.

Two stage-aware tweaks land here:
- `tampering_type_analysis.enabled` is forced to match `RUN_TAMPER` so the in-pipeline validator doesn't expect tamper outputs when the stage is off.
- `validation.require_outputs` flips to `'auto'` whenever any post-train stage is skipped, so `validate_experiment` only enforces the artefacts that the enabled stages actually produced.

In [ ]:
from src.config import load_config, deep_set, dump_config

config = load_config('configs/resnet_h95_config.yaml')

config = deep_set(config, 'data.root',                str(DATA_ROOT))
config = deep_set(config, 'data.tamper_metadata_dir', str(TAMPER_META))
config = deep_set(config, 'output.root_dir',          str(RESULTS_ROOT))
config = deep_set(config, 'output.run_dir',           'doctamper_resnet34_h95_35epochs_comparison')

# Resolve autotune setup: per-session overrides win over GPU-detected defaults.
batch_candidates_final = list(BATCH_CANDIDATES_OVERRIDE) if BATCH_CANDIDATES_OVERRIDE is not None else BATCH_CANDIDATES
memory_fraction_final  = float(MEMORY_FRACTION_OVERRIDE) if MEMORY_FRACTION_OVERRIDE  is not None else MEMORY_FRACTION
if BATCH_CANDIDATES_OVERRIDE is not None:
    print(f'gpu.batch_size_candidates overridden → {batch_candidates_final}')
if MEMORY_FRACTION_OVERRIDE is not None:
    print(f'gpu.auto_tune_memory_fraction overridden → {memory_fraction_final}')

config = deep_set(config, 'gpu.amp',                       True)
config = deep_set(config, 'gpu.amp_dtype',                 AMP_DTYPE)
config = deep_set(config, 'gpu.tf32',                      True)
config = deep_set(config, 'gpu.channels_last',             True)
config = deep_set(config, 'gpu.cudnn_benchmark',           True)
config = deep_set(config, 'gpu.torch_compile',             False)
config = deep_set(config, 'gpu.auto_tune_batch_size',      True)
config = deep_set(config, 'gpu.auto_tune_memory_fraction', memory_fraction_final)
config = deep_set(config, 'gpu.batch_size_candidates',     batch_candidates_final)

config = deep_set(config, 'training.batch_size',                 32)
config = deep_set(config, 'training.num_workers',                4)
config = deep_set(config, 'training.pin_memory',                 True)
config = deep_set(config, 'training.persistent_workers',         True)
config = deep_set(config, 'training.prefetch_factor',            4)
config = deep_set(config, 'training.gradient_accumulation_steps', 1)

if EPOCHS_OVERRIDE is not None:
    config = deep_set(config, 'training.epochs', int(EPOCHS_OVERRIDE))
    print(f'training.epochs overridden → {EPOCHS_OVERRIDE}')

# Stage-aware: tell the validator what to expect.
config = deep_set(config, 'tampering_type_analysis.enabled', bool(RUN_TAMPER))
all_post_train_on = RUN_EVAL and RUN_FAILURE and RUN_TAMPER
config = deep_set(config, 'validation.require_outputs', True if all_post_train_on else 'auto')

CONFIG_PATH = 'configs/resnet_h95_a100.yaml'
Path(CONFIG_PATH).write_text(dump_config(config), encoding='utf-8')
print(f'Wrote {CONFIG_PATH}')
print()
print(dump_config(config))

## 8. Smoke check — LMDB read + H95 stats + mask coverage  *(RUN_SMOKE)*

Walks the train folder and each test folder, reads two samples, prints H95 min/mean/max and mask positive ratio. Fails loudly if LMDB key schemes drift or any path is wrong.

In [ ]:
if RUN_SMOKE:
    from src.smoke import run_smoke
    reports = run_smoke(CONFIG_PATH, sample_count=2)
    print(f'\n✓ Smoke OK: {len(reports)} dataset(s) checked')
else:
    print('• RUN_SMOKE=False — skipped')

## 9. Training  *(RUN_TRAIN)*

When enabled, `run_train` performs:
1. `resolve_runtime` — stamps TF32, cuDNN benchmark, channels-last, AMP dtype.
2. `autotune_batch_size` — probes each candidate with a real forward+backward at `[B, 2, 512, 512]`, writes `batch_size_autotune.csv`, picks the largest under `auto_tune_memory_fraction`.
3. `_assert_model_contract` — forwards `[2, 2, 512, 512]`, asserts logits `[2, 1, 512, 512]`.
4. The full loop (epochs from `training.epochs` or `EPOCHS_OVERRIDE`) with AMP + grad clip + cosine LR; updates `train_metrics.csv`, `val_metrics.csv`, `plots/training_curves.png`, `checkpoints/{last,best}_*.pth` every epoch.

When disabled, the cell resolves `checkpoint` from `CHECKPOINT_OVERRIDE` or auto-discovers the best checkpoint on disk so downstream stages still have a model to score.

In [ ]:
from src.config import load_config as _load_cfg, resolve_output_paths as _resolve_paths

if RUN_TRAIN:
    from src.train import run_train
    checkpoint = str(run_train(CONFIG_PATH))
    print(f'\n✓ Training complete. Best checkpoint: {checkpoint}')
else:
    if CHECKPOINT_OVERRIDE:
        checkpoint = str(CHECKPOINT_OVERRIDE)
    else:
        _paths = _resolve_paths(_load_cfg(CONFIG_PATH))
        checkpoint = str(_paths['run'] / 'checkpoints' / 'best_model.pth')
    assert Path(checkpoint).exists(), (
        f'RUN_TRAIN=False but checkpoint not found: {checkpoint}. '
        'Set CHECKPOINT_OVERRIDE in § 3 to a valid *.pth, or run training first.'
    )
    print(f'• RUN_TRAIN=False — using existing checkpoint: {checkpoint}')

## 10. Evaluation  *(RUN_EVAL)*

`src.evaluate.run_evaluate` runs the model on TestingSet, FCD, SCD, writes per-image metrics, threshold sweeps, official 0.5 + best-threshold tables, diagnostic example panels, and bar-chart plots.

In [ ]:
if RUN_EVAL:
    from src.evaluate import run_evaluate
    eval_rows = run_evaluate(CONFIG_PATH, checkpoint)
    print(f'\n✓ Evaluation complete. {len(eval_rows)} test-set row(s).')
else:
    print('• RUN_EVAL=False — skipped')

## 11. Failure case analysis  *(RUN_FAILURE)*

`src.failure_analysis.run_failure_analysis` ranks the worst-K predictions per test set, writes summaries by category / reason / severity, and saves the worst-200 visualisations.

In [ ]:
if RUN_FAILURE:
    from src.failure_analysis import run_failure_analysis
    run_failure_analysis(CONFIG_PATH, checkpoint)
    print('\n✓ Failure analysis complete.')
else:
    print('• RUN_FAILURE=False — skipped')

## 12. Tamper-type analysis  *(RUN_TAMPER)*

`src.tampering_type_analysis.run_tampering_type_analysis` groups every prediction by the metadata-assigned tamper type (with heuristic fallback) and writes per-type metrics, threshold sweeps, and best-threshold tables.

In [ ]:
if RUN_TAMPER:
    from src.tampering_type_analysis import run_tampering_type_analysis
    run_tampering_type_analysis(CONFIG_PATH, checkpoint)
    print('\n✓ Tamper-type analysis complete.')
else:
    print('• RUN_TAMPER=False — skipped')

## 13. Summary + validate

Always-on wrap-up: writes the run-level `RES_SUMMARY.md` / `output.md`, then calls `validate_experiment`. With the stage-aware config from § 7, the validator only enforces the artefacts that the stages you ran actually produced.

In [ ]:
from src.output_summary import write_output_summary
from src.validate_experiment import validate_experiment

write_output_summary(CONFIG_PATH)
ok, errors = validate_experiment(CONFIG_PATH)
if ok:
    print('✓ validate_experiment: PASS')
else:
    print('⚠ validate_experiment: FAIL')
    for err in errors:
        print(f'  - {err}')
print(f'\nAll outputs under: {RESULTS_ROOT}')

## 14. Notes

### Where outputs land
Everything is written under `/content/drive/Othercomputers/MSI/results/doctamper_resnet34_h95_35epochs_comparison/`. Because this is a Drive-mirrored *"Other computer"* path, files appear on the MSI machine through Drive for Desktop as they're flushed.

Top-level artefacts created by the pipeline (relative to `RESULTS_ROOT`):
- `PLAN.md`, `RES_SUMMARY.md`, `output.md`, `model_summary.txt`, `resnet34_h95_evaluation_report.pdf`
- `doctamper_resnet34_h95_35epochs_comparison/`
  - `training.log`, `config_snapshot.yaml`, `config_resolved.yaml`, `batch_size_autotune.csv`, `gpu_profile.txt`
  - `train_metrics.csv`, `val_metrics.csv`, `plots/training_curves.png`
  - `splits/train_indices_seed42.csv`, `splits/val_indices_seed42.csv`
  - `checkpoints/best_model.pth`, `checkpoints/last_checkpoint.pth`
  - `summary_resnet34_h95_trained.csv`, `official_threshold_0.5_metrics.csv`, `best_threshold_metrics.csv`, `threshold_sweep.csv`, `per_image_metrics.csv`
  - `per_image_metrics/`, `threshold_sweeps/`, `examples/`
- `failure_case_analysis/` — failure summaries + worst-200 visualisations (only if § 11 ran)
- `tampering_type_analysis/` — per-tamper-type metrics + reports (only if § 12 ran)

### Resuming after disconnect
Re-run cells 1–6, then in § 9 manually resume from the last checkpoint instead of using the gated cell:
```python
from src.train import run_train
checkpoint = str(run_train(
    'configs/resnet_h95_a100.yaml',
    resume='/content/drive/Othercomputers/MSI/results/doctamper_resnet34_h95_35epochs_comparison/checkpoints/last_checkpoint.pth',
))
```
then continue from § 10.

### Re-running just the post-train stages on an existing checkpoint
Set in § 3:
```python
RUN_SMOKE = False
RUN_TRAIN = False
RUN_EVAL  = True   # or False
RUN_FAILURE = True # or False
RUN_TAMPER  = True # or False
CHECKPOINT_OVERRIDE = '/content/drive/Othercomputers/MSI/results/doctamper_resnet34_h95_35epochs_comparison/checkpoints/best_model.pth'
```
Run everything top to bottom.

### Pushing local debugging edits back to Drive
If you edit anything in `/content/Train_resnet_server/src/` for debugging, copy it back manually before ending the session, e.g.:
```bash
!cp /content/Train_resnet_server/src/<file>.py /content/drive/Othercomputers/MSI/Approach/RESNET34/Train_resnet_server/src/<file>.py
```
Section 5's `cp -r` runs only when `/content/Train_resnet_server` is missing, so a stale local copy survives until the runtime restarts.

### Pushing the batch size higher
If a full epoch at the autotune-selected batch finishes with peak VRAM well below 36 GB (see `gpu_max_reserved_gb` in `train_metrics.csv`), rerun training with a more aggressive ceiling:
```python
config = deep_set(config, 'gpu.batch_size_candidates',     [64, 80, 96])
config = deep_set(config, 'gpu.auto_tune_memory_fraction', 0.86)
```